In [2]:
import pandas as pd

In [ ]:
from groq import Groq

client = Groq(
    api_key="YOUR_GROQ_API_KEY"
)

In [4]:

df = pd.read_csv("IT.csv")  
df.head()

,Unnamed: 0,Body,Department,Priority,Tags
0,0,"Dear Customer Support Team,I am writing to rep...",Technical Support,high,"['Account', 'Disruption', 'Outage', 'IT', 'Tec..."
1,1,"Dear Customer Support Team,I hope this message...",Returns and Exchanges,medium,"['Product', 'Feature', 'Tech Support']"
2,2,"Dear Customer Support Team,I hope this message...",Billing and Payments,low,"['Billing', 'Payment', 'Account', 'Documentati..."
3,3,"Dear Support Team,I hope this message reaches ...",Sales and Pre-Sales,medium,"['Product', 'Feature', 'Feedback', 'Tech Suppo..."
4,4,"Dear Customer Support,I hope this message reac...",Technical Support,high,"['Feature', 'Product', 'Documentation', 'Feedb..."


In [5]:
df.columns

Index(['Unnamed: 0', 'Body', 'Department', 'Priority', 'Tags'], dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   str    
 2   Customer Email                8469 non-null   str    
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   str    
 5   Product Purchased             8469 non-null   str    
 6   Date of Purchase              8469 non-null   str    
 7   Ticket Type                   8469 non-null   str    
 8   Ticket Subject                8469 non-null   str    
 9   Ticket Description            8469 non-null   str    
 10  Ticket Status                 8469 non-null   str    
 11  Resolution                    2769 non-null   str    
 12  Ticket Priority               8469 non-null   str    
 13  Ticket Channel

In [6]:
df = df[['Body', 'Department']]
df = df.sample(30, random_state=42).reset_index(drop=True)

print(df["Department"].unique())
df.head()

<ArrowStringArray>
[              'Technical Support',                'Customer Service',
                 'Product Support',            'Billing and Payments',
                 'General Inquiry',           'Returns and Exchanges',
 'Service Outages and Maintenance']
Length: 7, dtype: str


,Body,Department
0,The encryption of health data has suddenly sto...,Technical Support
1,There was a system outage on my MacBook Pro th...,Technical Support
2,While integrating a data analytics platform wi...,Technical Support
3,"Customer Support, please address a critical is...",Customer Service
4,I am encountering a login problem in the mobil...,Product Support


In [7]:
ticket = df.loc[0, "Body"]

categories = "\n".join(df["Department"].unique())

prompt = f"""
You are a support ticket classifier.

Possible categories:
{categories}

Read the ticket and return ONLY:
1. Best category
2. Top 3 most likely categories

Ticket:
{ticket}
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=0
)

print(response.choices[0].message.content)

1. Best category: Technical Support
2. Top 3 most likely categories: 
   1. Technical Support
   2. Product Support
   3. Service Outages and Maintenance


In [10]:
predictions = []

categories = "\n".join(df["Department"].unique())

for ticket in df["Body"][:10]:

    prompt = f"""
You are a support ticket classifier.

Possible categories:
{categories}

Examples:

Ticket: My internet connection keeps disconnecting every few minutes.
Category: Technical Support

Ticket: I was charged twice on my credit card this month.
Category: Billing and Payments

Ticket: I want to return the product because it arrived damaged.
Category: Returns and Exchanges

Ticket: I need information about your premium subscription plan.
Category: General Inquiry

Ticket: My order has not arrived yet. Can you help me?
Category: Customer Service

Ticket: The software installation keeps failing with an error.
Category: Product Support

Now classify the following ticket.

Return ONLY the category.

Ticket:
{ticket}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    predictions.append(response.choices[0].message.content.strip())

df_result = df[:10].copy()
df_result["Predicted"] = predictions

df_result[["Department", "Predicted"]]

,Department,Predicted
0,Technical Support,Technical Support
1,Technical Support,Technical Support
2,Technical Support,Technical Support
3,Customer Service,Technical Support
4,Product Support,Technical Support
5,Billing and Payments,Billing and Payments
6,General Inquiry,Technical Support
7,Product Support,Technical Support
8,Technical Support,Technical Support
9,Product Support,General Inquiry


In [11]:
correct = (df_result["Department"] == df_result["Predicted"]).sum()
accuracy = correct / len(df_result)

print(f"Correct Predictions: {correct}/{len(df_result)}")
print(f"Accuracy: {accuracy:.2%}")

Correct Predictions: 5/10
Accuracy: 50.00%


In [12]:
ticket = df.loc[0, "Body"]

categories = "\n".join(df["Department"].unique())

prompt = f"""
You are a support ticket classifier.

Possible categories:
{categories}

For the following ticket, return the TOP 3 most likely categories in order of confidence.

Return ONLY like this:

1. Category
2. Category
3. Category

Ticket:
{ticket}
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print(response.choices[0].message.content)

1. Technical Support
2. Product Support
3. Service Outages and Maintenance


In [13]:
# Zero-shot predictions
zero_shot = []

categories = "\n".join(df["Department"].unique())

for ticket in df["Body"][:10]:

    prompt = f"""
You are a support ticket classifier.

Possible categories:
{categories}

Return ONLY the category name.

Ticket:
{ticket}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    zero_shot.append(response.choices[0].message.content.strip())

# Compare with few-shot predictions
comparison = df[:10].copy()
comparison["Zero-shot"] = zero_shot
comparison["Few-shot"] = predictions

comparison[["Department", "Zero-shot", "Few-shot"]]

,Department,Zero-shot,Few-shot
0,Technical Support,Technical Support,Technical Support
1,Technical Support,Technical Support,Technical Support
2,Technical Support,Technical Support,Technical Support
3,Customer Service,Technical Support,Technical Support
4,Product Support,Technical Support,Technical Support
5,Billing and Payments,Billing and Payments,Billing and Payments
6,General Inquiry,Technical Support,Technical Support
7,Product Support,Technical Support,Technical Support
8,Technical Support,Technical Support,Technical Support
9,Product Support,Product Support,General Inquiry


## Summary

- Loaded and preprocessed a support ticket dataset.
- Used Groq Llama 3.3 for automatic ticket tagging.
- Performed zero-shot and few-shot classification.
- Compared the predictions of both approaches.
- Generated the top 3 most probable tags for a support ticket.
- Few-shot prompting produced similar results on this sample, while demonstrating prompt engineering as required.
